<a href="https://colab.research.google.com/github/IreneDeNevi/nlp_lime/blob/main/nlp_lime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LIME for Traditional Text Classification

This section demonstrates how to use LIME (`LimeTextExplainer`) to interpret predictions from a traditional machine learning model (TF-IDF + Multinomial Naive Bayes) applied to text classification. We use the 20 Newsgroups dataset as an example.

### Install Required Libraries

First, we install the `lime` library, along with `scikit-learn`, `numpy`, `pandas`, and `matplotlib` which are commonly used for data manipulation, machine learning, and visualization.

### Load Data and Train Model

We load a subset of the 20 Newsgroups dataset, preprocess the text using `TfidfVectorizer`, and train a `MultinomialNB` classifier. A pipeline is used to combine these steps. Finally, we select a random text sample from the test set and display its content along with its true category. This sample will be used for LIME explanation.

### Generate and Visualize LIME Explanation

Here, we initialize `LimeTextExplainer` with the class names. We then use `explain_instance` to get an explanation for the `sample_text` using the `pipeline.predict_proba` function. The `num_features` parameter specifies how many top features (words) should be highlighted. The `exp.show_in_notebook(text=True)` displays the explanation directly in the notebook, highlighting words that contribute positively (green) or negatively to the predicted class. The `exp.as_pyplot_figure()` generates a bar chart showing the feature importance.

## LIME for Large Language Models (LLM)

This section demonstrates how to apply LIME to explain predictions from a pre-trained Large Language Model (LLM) from Hugging Face for sentiment analysis.

### Install Required Libraries for LLM

We install `lime`, `transformers` (for LLMs), `torch` (as a backend for transformers), `datasets`, `numpy`, `pandas`, and `matplotlib`.

### Import Libraries for LLM Explanation

Importing the necessary Python libraries for this section, including `torch`, `numpy`, `matplotlib`, `transformers` for the LLM pipeline, and `LimeTextExplainer`.

### Load LLM and Make Initial Prediction

We load a pre-trained `distilbert-base-uncased-finetuned-sst-2-english` model for text classification (sentiment analysis) from Hugging Face. A custom `predict_proba` function is defined to work with LIME, which requires probability scores for all classes. We then select a `sample_text` and display its initial prediction from the model.

### Generate and Visualize LIME Explanation for LLM

An `LimeTextExplainer` is initialized for the LLM. The `explain_instance` method is called with the `sample_text` and our `predict_proba` function to generate an explanation. The explanation is then displayed in the notebook, showing which words contribute to the positive or negative sentiment prediction. A bar plot further visualizes these contributions.

## LIME for GANs-like Tabular Data

This section illustrates how LIME (`LimeTabularExplainer`) can be used to interpret the decisions of a classifier that acts like a discriminator in a Generative Adversarial Network (GAN) setting, but for tabular data.

### Simulate Data, Train Classifier, and Generate LIME Explanation

We define a `conditional_generator` function that simulates generating outputs based on input conditions. We then create random `conditions` and `generated_outputs`, and based on these outputs, create binary `labels`. A `LogisticRegression` model is trained on these conditions and labels, acting as a 'discriminator' that predicts the likelihood of an output being 'high' or 'low' given the conditions.

A `LimeTabularExplainer` is initialized, providing the training data, feature names (`cond_1`, `cond_2`), and class names (`low_output`, `high_output`). We select a specific `sample_condition` to explain and use `explain_instance` with the classifier's `predict_proba` method. The `num_features` is set to 2 to explain the impact of both conditions. The explanation is visualized as a bar chart, showing the contribution of each condition to the predicted probability, and the predicted probability for the sample is printed.

## SHAP for Large Language Models (LLM)

This section demonstrates how to use SHAP (SHapley Additive exPlanations) to interpret predictions from a pre-trained Large Language Model (LLM) from Hugging Face for sentiment analysis. SHAP provides a game-theoretic approach to explain the output of any machine learning model.

### Install Required Libraries

We install `shap` along with `scikit-learn`, `numpy`, and `pandas`.

### Load LLM, Define Prediction Function, and Generate SHAP Explanation

First, we load the same Hugging Face sentiment analysis pipeline as in the LIME example. We then define a simple wrapper function `f` that takes text inputs and returns the model's prediction scores, which is required by SHAP. We define a list of `texts` to explain. `shap.Explainer(classifier)` creates a SHAP explainer specifically designed to work with Hugging Face pipelines. `explainer(texts)` computes the SHAP values for each word in each text. Finally, `shap.plots.text(shap_values[0])` visualizes the SHAP explanation for the first sample, showing how each word contributes to the overall sentiment prediction.

In [ ]:
# Install necessary libraries
!pip install lime scikit-learn numpy pandas matplotlib



In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups
from lime.lime_text import LimeTextExplainer

# Load dataset (20 Newsgroups)
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'talk.politics.mideast']
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

# Extract text data and labels
X_train, y_train = newsgroups_train.data, newsgroups_train.target
X_test, y_test = newsgroups_test.data, newsgroups_test.target

# Create a TF-IDF vectorizer and Naive Bayes classifier
vectorizer = TfidfVectorizer(stop_words='english')
classifier = MultinomialNB()

# Create a pipeline
pipeline = make_pipeline(vectorizer, classifier)

# Train the model
pipeline.fit(X_train, y_train)

# Define class names for readability
class_names = newsgroups_train.target_names

# Select a random instance from the test dataset
idx = np.random.randint(0, len(X_test))
sample_text = X_test[idx]
true_label = y_test[idx]

print("\n**Original Text Sample:**\n")
print(sample_text)
print("\n**True Category:**", class_names[true_label])




In [ ]:
# Initialize LIME explainer
explainer = LimeTextExplainer(class_names=class_names)


# Generate explanation for the selected instance
#exp = explainer.explain_instance(sample_text, pipeline.predict_proba, num_features=10)

exp = explainer.explain_instance(sample_text, pipeline.predict_proba,num_features=10)

# Display the explanation
print("\n**LIME Explanation for Text Classification:**")
exp.show_in_notebook(text=True)

# Plot feature importance
fig = exp.as_pyplot_figure()
plt.show()

# **LLM Lime Explainer**

In [ ]:
# Install necessary libraries
!pip install lime transformers torch datasets numpy pandas matplotlib

In [ ]:
# Import required libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline
from lime.lime_text import LimeTextExplainer

In [ ]:
# Load a pre-trained text classification model from Hugging Face
classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

# Function to predict probabilities for LIME
def predict_proba(texts):
    results = classifier(texts, return_all_scores=True)  # Get all class probabilities
    return np.array([[score['score'] for score in result] for result in results])

# Define class names (positive/negative sentiment)
class_names = ["Negative", "Positive"]

# Select a sample text for explanation
sample_text = "I absolutely love this movie! The story was fantastic and the acting was top-notch."

# Print the original prediction
print("\n**Original Text Sample:**")
print(sample_text)
print("\n**Model Prediction:**", classifier(sample_text))


In [ ]:
# Initialize LIME explainer
explainer = LimeTextExplainer(class_names=class_names)

# Generate explanation for the selected text instance
exp = explainer.explain_instance(sample_text, predict_proba, num_features=10)

# Display explanation
print("\n**LIME Explanation for BERT Prediction:**")
exp.show_in_notebook(text=True)

# Plot feature importance
fig = exp.as_pyplot_figure()
plt.show()

In [ ]:
# Install required libraries
!pip install lime scikit-learn numpy matplotlib


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from lime.lime_tabular import LimeTabularExplainer

# Step 1: Simulate a conditional generator (analogous to conditional GAN output)
def conditional_generator(conditions):
    outputs = np.sin(conditions[:, 0]) + np.cos(conditions[:, 1]) + np.random.normal(0, 0.1, size=(conditions.shape[0],))
    return outputs

# Step 2: Create simulated conditions and generated outputs
np.random.seed(0)
conditions = np.random.uniform(-3, 3, (500, 2))
generated_outputs = conditional_generator(conditions)

# Step 3: Train a simple classifier (discriminator-like)
labels = (generated_outputs > 0.5).astype(int)
classifier = LogisticRegression()
classifier.fit(conditions, labels)

# Step 4: Use LIME to explain predictions based on input conditions
explainer = LimeTabularExplainer(training_data=conditions, feature_names=["cond_1", "cond_2"],
                                 class_names=["low_output", "high_output"], mode="classification")

# Step 5: Choose a sample condition to explain
sample_idx = 42
sample_condition = conditions[sample_idx]
exp = explainer.explain_instance(sample_condition, classifier.predict_proba, num_features=2)

# Step 6: Show explanation
fig = exp.as_pyplot_figure()
plt.show()

predicted_prob = classifier.predict_proba(sample_condition.reshape(1, -1))[0, 1]
print(f"Condition: {sample_condition}, Predicted Probability (High Output): {predicted_prob:.3f}")


**gans**

In [ ]:
# Install required libraries
!pip install lime scikit-learn numpy matplotlib



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from lime.lime_tabular import LimeTabularExplainer

# Step 1: Simulate a conditional generator (analogous to conditional GAN output)
def conditional_generator(conditions):
    outputs = np.sin(conditions[:, 0]) + np.cos(conditions[:, 1]) + np.random.normal(0, 0.1, size=(conditions.shape[0],))
    return outputs

# Step 2: Create simulated conditions and generated outputs
np.random.seed(0)
conditions = np.random.uniform(-3, 3, (500, 2))
generated_outputs = conditional_generator(conditions)

# Step 3: Train a simple classifier (discriminator-like)
labels = (generated_outputs > 0.5).astype(int)
classifier = LogisticRegression()
classifier.fit(conditions, labels)




In [ ]:
# Step 4: Use LIME to explain predictions based on input conditions
explainer = LimeTabularExplainer(training_data=conditions, feature_names=["cond_1", "cond_2"],
                                 class_names=["low_output", "high_output"], mode="classification")

# Step 5: Choose a sample condition to explain
sample_idx = 42
sample_condition = conditions[sample_idx]
exp = explainer.explain_instance(sample_condition, classifier.predict_proba, num_features=2)

# Step 6: Show explanation
fig = exp.as_pyplot_figure()
plt.show()

predicted_prob = classifier.predict_proba(sample_condition.reshape(1, -1))[0, 1]
print(f"Condition: {sample_condition}, Predicted Probability (High Output): {predicted_prob:.3f}")

**Shapley**

In [ ]:
# Install SHAP and required libraries
!pip install shap scikit-learn numpy pandas

In [ ]:
# Install necessary libraries
!pip install shap transformers torch

import shap
import numpy as np
from transformers import pipeline
import matplotlib.pyplot as plt

# Load a Hugging Face sentiment analysis pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Define a prediction function for SHAP
def f(x):
    return np.array([output["score"] for output in classifier(x)]).reshape(-1, 1)

# Example texts to explain
texts = [
    "I love this movie. It's fantastic!",
    "This was the worst experience I've ever had.",
    "The product is okay, but could be better.",
]

# Use SHAP's built-in explainer for Hugging Face pipelines
explainer = shap.Explainer(classifier)

# Compute SHAP values
shap_values = explainer(texts)

# Plot SHAP explanation for the first sample
shap.plots.text(shap_values[0])
